# Second 1.0 — Stage T3: SFT v2 Iteration and Capacity Ablations

This notebook trains and evaluates the **Second-1.0-2B-SFT-v2** model on Google Colab with an **NVIDIA L4 or A100 GPU**.
It implements Stage T3 of `TRAINING_PLAN.md` across the controlled ablation matrix.

### Stage T3 Ablation Matrix

| Exp ID | Dataset Source & Size | Adapter / Plasticity | Loss Masking | Purpose |
|---|---|---|---|---|
| `SFT-A1` | 3,000 verified teacher (`thinking: off`) | LoRA `r=32, alpha=32` | Answer-only | Test LIMA/LIMO small gold dataset hypothesis. |
| `SFT-A3` | 2,850 verified teacher + 350 gold human (3,200 total) | LoRA `r=32, alpha=32` | Answer-only | Test human stylistic variety. |
| `SFT-A4` | 3,200 verified records | LoRA `r=64, alpha=128` | Answer-only | Test higher adapter capacity. |
| `SFT-A5` | 3,200 verified records | Full Parameter (8-bit AdamW, `lr=2e-5`) | Answer-only | Test full fine-tuning at 2B scale. |
| `SFT-A6` | 3,200 dual-mode (50% thinking, 50% non-thinking) | Best plasticity from A3–A5 | `<think>` + Answer | Dual-mode SFT checkpoint for Stage T4/T5. |

### Stage T3 Pass Gates (Evaluated on `eval/devset-1.0.jsonl`)
- **Dev Hard Fail Rate**: &le; 6.0%
- **Dev In-Band Length**: &ge; 75.0% (both `short` and `long` slices &ge; 70.0%)
- **Dev Key Point Score**: &ge; 0.65
- **Evaluation Rule**: Never tune or train on `eval/testset-1.0.jsonl`. Evaluate on `eval/devset-1.0.jsonl` only.

In [ ]:
# Step 1: Verify GPU Environment
!nvidia-smi

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemError("No GPU detected. In Colab menu: Runtime -> Change runtime type -> L4 or A100 GPU.")

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
bf16_supported = torch.cuda.is_bf16_supported()
print(f"GPU: {device_name} ({vram_gb:.1f} GB VRAM, bf16={bf16_supported})")


In [ ]:
# Step 2: Mount Google Drive for Persistent Storage
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = Path("/content/drive/MyDrive/second")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted: {DRIVE_DIR}")
except Exception as exc:
    print(f"Drive mount notice: {exc}")
    DRIVE_DIR = Path("/content/second_local")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Step 3: Extract Stage T3 SFT Bundle or Setup Local Repo
import shutil
import zipfile

WORK_DIR = Path("/content/Second")
WORK_DIR.mkdir(parents=True, exist_ok=True)

bundle_locations = [
    DRIVE_DIR / "data" / "phase3_sft_bundle.zip",
    Path("/content/phase3_sft_bundle.zip"),
    Path("data/processed/phase3_sft_bundle.zip"),
]

bundle_path = None
for loc in bundle_locations:
    if loc.is_file():
        bundle_path = loc
        break

if bundle_path and bundle_path.is_file():
    print(f"Extracting bundle from {bundle_path} into {WORK_DIR}...")
    with zipfile.ZipFile(bundle_path, "r") as zf:
        zf.extractall(WORK_DIR)
    os.chdir(str(WORK_DIR))
    print(f"Bundle extracted. Working directory: {os.getcwd()}")
elif Path("training/sft_v2_data.py").is_file():
    print("Working in existing repository checkout.")
else:
    raise FileNotFoundError("phase3_sft_bundle.zip not found. Please upload it to Google Drive or /content.")

In [ ]:
# Step 4: Install Dependencies
!pip install -q -U torchao
!pip install -q transformers peft accelerate bitsandbytes chess pyyaml


In [ ]:
# Step 5: Select Ablation Recipe Configuration
# Choose one: SFT-A1 (LoRA r=32), SFT-A4 (LoRA r=64), SFT-A5 (Full Parameter), SFT-A6 (Dual Mode)
EXPERIMENT_ID = "SFT-A1"  # Change to SFT-A4, SFT-A5, or SFT-A6 as desired

config_map = {
    "SFT-A1": "training/configs/sft_v2_a1_lora32.yaml",
    "SFT-A3": "training/configs/sft_v2_a1_lora32.yaml",
    "SFT-A4": "training/configs/sft_v2_a4_lora64.yaml",
    "SFT-A5": "training/configs/sft_v2_a5_full.yaml",
    "SFT-A6": "training/configs/sft_v2_a6_dual.yaml",
}

from training.sft_v2_data import load_sft_v2_config
config_file = config_map[EXPERIMENT_ID]
cfg = load_sft_v2_config(config_file)
print(f"Loaded recipe configuration: {config_file}")
print(f"Experiment: {EXPERIMENT_ID} | Model: {cfg.model_name} | Method: {cfg.method} | LR: {cfg.learning_rate}")
print(f"Training dataset: {cfg.train_data_path}")
print(f"Devset (Protected): {cfg.devset_path}")

In [ ]:
# Step 6: Load Base Model and Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

print(f"Loading base model: {cfg.base_model} in bf16...")
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if cfg.method == "lora":
    model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=cfg.target_modules,
        bias="none",
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
else:
    # Full parameter fine-tuning
    model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    print("Configured Full Parameter Fine-Tuning.")

In [ ]:
# Step 7: Prepare Datasets with Answer-Only Loss Masking
from training.sft_v2_data import (
    SFTv2DataCollator,
    load_sft_v2_records,
    tokenize_sft_v2_record,
)
from torch.utils.data import Dataset

raw_train_records = load_sft_v2_records(cfg.train_data_path, max_examples=cfg.max_examples)
raw_val_records = load_sft_v2_records(cfg.val_data_path, max_examples=500)

class SFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_seq_length, loss_on_answer_only):
        self.features = [
            tokenize_sft_v2_record(
                r,
                encode_fn=tokenizer.encode,
                max_seq_length=max_seq_length,
                loss_on_answer_only=loss_on_answer_only,
            )
            for r in records
        ]
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx]

train_dataset = SFTDataset(
    raw_train_records,
    tokenizer,
    max_seq_length=cfg.max_seq_length,
    loss_on_answer_only=cfg.loss_on_answer_only,
)
val_dataset = SFTDataset(
    raw_val_records,
    tokenizer,
    max_seq_length=cfg.max_seq_length,
    loss_on_answer_only=cfg.loss_on_answer_only,
)

collator = SFTv2DataCollator(pad_token_id=tokenizer.pad_token_id)
print(f"Prepared {len(train_dataset)} training examples and {len(val_dataset)} validation examples.")

In [ ]:
# Step 8: Run SFT Training with Checkpointing
from transformers import Trainer, TrainingArguments

checkpoint_dir = Path(cfg.checkpoint_dir)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Enable gradient checkpointing to drastically cut VRAM usage during training
model.gradient_checkpointing_enable()
model.config.use_cache = False  # Recommended with gradient checkpointing

batch_size = 2
grad_accum = 4  # Keeps effective batch size at 8 (2 * 4 == 8)
warmup_steps = int(len(train_dataset) // (batch_size * grad_accum) * cfg.num_train_epochs * cfg.warmup_ratio)

training_args = TrainingArguments(
    output_dir=str(checkpoint_dir),
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    gradient_checkpointing=True,
    learning_rate=cfg.learning_rate,
    lr_scheduler_type=cfg.lr_scheduler_type,
    warmup_steps=warmup_steps,
    logging_steps=cfg.logging_steps,
    save_strategy=cfg.save_strategy,
    save_steps=cfg.save_steps,
    eval_strategy=cfg.eval_strategy,
    eval_steps=cfg.eval_steps,
    bf16=cfg.bf16,
    save_total_limit=3,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

print(f"Starting SFT v2 training for {cfg.num_train_epochs} epochs (batch_size={batch_size}, grad_accum={grad_accum}, effective={batch_size*grad_accum})...")
trainer.train()

final_model_dir = checkpoint_dir / "final_adapter"
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))
print(f"Training complete! Saved final checkpoint to {final_model_dir}")


In [ ]:
# Step 9: Generate Greedy Outputs on Development Set (eval/devset-1.0.jsonl)
import json

model.eval()
tokenizer.padding_side = "left"

devset_file = Path(cfg.devset_path)
with open(devset_file, "r", encoding="utf-8") as f:
    dev_items = [json.loads(line) for line in f if line.strip()]

print(f"Generating greedy completions on {len(dev_items)} devset items...")
output_records = []
batch_size = 8

for start_idx in range(0, len(dev_items), batch_size):
    batch = dev_items[start_idx : start_idx + batch_size]
    prompts = [
        f"<|im_start|>system\nYou are Second, a chess assistant. "
        f"You explain chess positions and moves clearly using the provided facts.<|im_end|>\n"
        f"<|im_start|>user\n{item['user_message']}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n"
        for item in batch
    ]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    for item, out_ids, in_ids in zip(batch, outputs, inputs["input_ids"]):
        gen_ids = out_ids[len(in_ids):]
        comp = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        output_records.append({
            "id": item["id"],
            "completion": comp,
        })

dev_output_path = Path(f"eval/results/sft_v2_{EXPERIMENT_ID.lower()}_dev_outputs.jsonl")
dev_output_path.parent.mkdir(parents=True, exist_ok=True)
with open(dev_output_path, "w", encoding="utf-8") as f:
    for r in output_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved {len(output_records)} completions to {dev_output_path}")

In [ ]:
# Step 10: Run Stage T3 Gate Evaluation and Verification
!python -m training.evaluate_sft_v2 \
    --devset eval/devset-1.0.jsonl \
    --outputs eval/results/sft_v2_{EXPERIMENT_ID.lower()}_dev_outputs.jsonl \
    --experiment-id {EXPERIMENT_ID} \
    --reports-dir eval/reports

# Backup outputs and reports to Google Drive
drive_results_dir = DRIVE_DIR / "results" / EXPERIMENT_ID.lower()
drive_results_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(dev_output_path, drive_results_dir / dev_output_path.name)

report_path = Path(f"eval/reports/stage_t3_{EXPERIMENT_ID.lower()}_report.md")
if report_path.is_file():
    shutil.copy2(report_path, drive_results_dir / report_path.name)
    print(f"\nSaved evaluation report to Google Drive: {drive_results_dir / report_path.name}")

print(f"\nStage T3 SFT v2 ({EXPERIMENT_ID}) Complete!")